# Revisión de los datos de la DB de Hedwix

Este notebook se conecta a MongoDB con `MONGODB_URI`, igual que la app, y revisa que los datos estén bien:

1. **Conexión y conteos**: qué colecciones hay y cuántos documentos tienen.
2. **Panorama de `noticias`**: noticias por fuente y por día, y las corridas del funnel.
3. **Chequeos de calidad**: cada chequeo dice ✅ o ❌. Al final hay una tabla resumen.
4. **Índices**: que existan y que las consultas del MCP y el agent sí los usen.
5. **Muestras**: últimas noticias, búsqueda de texto y una noticia completa.

Corre todo con *Run → Run All Cells*. Para correrlo sin abrir Jupyter, ver `db/testing/README.md`.

In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Funciona igual en Docker (db/ montado en /app, el repo en /repo) que abierto directo desde db/testing/.
DB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hedwix_db").is_dir())   # carpeta db/
RAIZ = next(p for p in [DB.parent, Path("/repo")] if (p / "funnel").is_dir())           # raíz del repo
sys.path[:0] = [str(DB), str(RAIZ / "funnel")]
if "MONGODB_URI" not in os.environ:
    # Fuera de Docker: se toma la URI del .env de la raíz y se conecta por 127.0.0.1 en lugar del host "mongo"
    from dotenv import dotenv_values
    os.environ["MONGODB_URI"] = dotenv_values(RAIZ / ".env")["MONGODB_URI"].replace("@mongo:", "@127.0.0.1:")

from hedwix_db import obtener_db
from hedwix_db.noticias import a_item, buscar, mas_populares, recientes
from fuentes_curadas import id_de_url      # la misma función con la que el funnel calcula el id (funnel/)

pd.set_option("display.max_colwidth", 80)
db = obtener_db()
resultados = []                            # (chequeo, problemas encontrados) para el resumen final

def chequeo(nombre, problemas, detalle=None):
    """Registra un chequeo. `problemas` es cuántos documentos fallan (0 = bien)."""
    resultados.append({"chequeo": nombre, "problemas": problemas, "ok": problemas == 0})
    print(("✅" if problemas == 0 else "❌"), nombre, "" if problemas == 0 else f"→ {problemas} con problemas")
    if problemas and detalle is not None:
        display(detalle)

## 1. Conexión y conteos

In [ ]:
print("Servidor MongoDB", db.client.server_info()["version"], "| base:", db.name)
conteos = pd.DataFrame(
    [{"colección": nombre, "documentos": db[nombre].estimated_document_count()}
     for nombre in sorted(db.list_collection_names())]
)
conteos

In [ ]:
print("Migraciones aplicadas:")
pd.DataFrame(db["_migraciones"].find().sort("_id"))

## 2. Panorama de `noticias`

Se cargan todas las noticias **sin `content`**, que es el campo pesado, para analizarlas con pandas.

In [ ]:
noticias = pd.DataFrame(db.noticias.aggregate([
    {"$addFields": {"largo_content": {"$strLenCP": "$content"}}},   # el largo se calcula en Mongo
    {"$project": {"content": 0}},
]))
print(len(noticias), "noticias |", noticias["source"].nunique(), "fuentes |",
      "publicadas entre", noticias["published"].min(), "y", noticias["published"].max())
noticias.groupby(["origin", "source_type"]).size().rename("noticias").to_frame()

In [ ]:
por_fuente = noticias.groupby("source").agg(
    noticias=("_id", "size"),
    con_score=("score", lambda s: s.notna().sum()),
    content_vacio=("largo_content", lambda s: (s == 0).sum()),
    content_mediana=("largo_content", "median"),
    mas_reciente=("published", "max"),
).sort_values("noticias", ascending=False)
por_fuente

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(14, 6))
por_fuente["noticias"].sort_values().plot.barh(ax=izq, title="Noticias por fuente")
(noticias.set_index("published").resample("D").size()
 .plot(ax=der, marker="o", title="Noticias por día de publicación"))
der.set_xlabel("")
plt.tight_layout()

In [ ]:
print("Corridas del funnel (cada fetched_at distinto es una corrida o una carga):")
noticias.groupby("fetched_at").size().rename("noticias nuevas").to_frame()

## 3. Chequeos de calidad

Si todo está bien, todos salen ✅. Cuando un chequeo falla, debajo aparecen los documentos con problema.

In [ ]:
# 3.1 Todo cumple el validador actual de la colección. Esto atrapa documentos que entraron
#     antes de un cambio de esquema, porque el validador sólo revisa al insertar o actualizar.
esquema = db.command("listCollections", filter={"name": "noticias"})["cursor"]["firstBatch"][0]["options"]["validator"]["$jsonSchema"]
fuera = list(db.noticias.find({"$nor": [{"$jsonSchema": esquema}]}, {"_id": 1, "source": 1, "title": 1}))
chequeo("Todas las noticias cumplen el $jsonSchema", len(fuera), pd.DataFrame(fuera))

# 3.2 El _id es el sha1 de la URL normalizada, calculado igual que en el funnel
mal_id = noticias[noticias["_id"] != noticias["url"].map(id_de_url)]
chequeo("_id = sha1(url normalizada)", len(mal_id), mal_id[["_id", "source", "url"]])

# 3.3 No hay dos noticias con la misma URL normalizada
repetidas = noticias[noticias["url"].map(id_de_url).duplicated(keep=False)]
chequeo("Sin URLs repetidas", len(repetidas), repetidas[["_id", "source", "url"]])

In [ ]:
ahora = datetime.now(timezone.utc)

# 3.4 Fechas: guardadas como fecha (no texto), sin noticias del futuro y sin publicadas después de traídas
no_fecha = db.noticias.count_documents({"$or": [{"published": {"$not": {"$type": "date"}}},
                                                 {"fetched_at": {"$not": {"$type": "date"}}}]})
chequeo("published y fetched_at son Date", no_fecha)
futuras = noticias[noticias["published"] > ahora]
chequeo("Ninguna publicada en el futuro", len(futuras), futuras[["source", "title", "published"]])
# Los rankings (OpenRouter, HF trending) usan como `published` la hora de la consulta, que es unos
# segundos después de `fetched_at` (el funnel la fija al empezar la corrida). Por eso hay tolerancia.
TOLERANCIA = pd.Timedelta(minutes=5)
despues = noticias[noticias["published"] > noticias["fetched_at"] + TOLERANCIA]
chequeo("published <= fetched_at (+5 min)", len(despues), despues[["source", "title", "published", "fetched_at"]])

# 3.5 score sólo en las fuentes que tienen popularidad (Hacker News y HF Daily Papers)
CON_SCORE = {"Hacker News", "HF Daily Papers"}
score_raro = noticias[noticias["score"].notna() != noticias["source"].isin(CON_SCORE)]
chequeo("score sólo en Hacker News / HF Daily Papers", len(score_raro), score_raro[["source", "title", "score"]])

# 3.6 Títulos sin HTML
con_html = noticias[noticias["title"].str.contains(r"<[a-zA-Z/][^>]*>", regex=True)]
chequeo("Títulos sin HTML", len(con_html), con_html[["source", "title"]])

# 3.7 Ninguna fuente trae *todas* sus notas sin texto. Si pasa, probablemente la fuente cambió y sólo
#     llegan títulos. HF trending nunca trae texto (ver ESQUEMA.md), así que no cuenta.
SIN_TEXTO_ESPERADO = {"Hugging Face: modelos trending"}
otras = noticias[~noticias["source"].isin(SIN_TEXTO_ESPERADO)]
sin_texto = otras.groupby("source")["largo_content"].max().loc[lambda largo: largo == 0]
chequeo("Ninguna fuente sin texto en todas sus notas", len(sin_texto), sin_texto.rename("largo máximo").to_frame())

# Las notas sueltas sin texto se muestran pero no fallan: por ejemplo, un link de Hacker News
# a un sitio que bloquea la descarga (Reuters responde 401) y no trae texto propio.
vacias = otras[otras["largo_content"] == 0]
print(f"ℹ️  {len(vacias)} notas sueltas sin texto:")
display(vacias[["source", "title", "url"]])

In [ ]:
# 3.8 No sólo el formato, también qué tan completo es el texto. Sólo informa, no falla:
#     las notas que el funnel no pudo descargar quedan con el texto corto de la fuente.
DESCARGADAS = ["OpenAI Blog", "Hugging Face Blog", "TechCrunch AI", "Databricks Blog", "Google DeepMind Blog",
               "Mistral News", "Anthropic News", "Meta AI Blog", "Hacker News"]
cortas = noticias[noticias["source"].isin(DESCARGADAS) & (noticias["largo_content"] < 600)]
print(f"ℹ️  {len(cortas)} notas de fuentes que se descargan quedaron con menos de 600 caracteres "
      "(probablemente la descarga del texto completo falló):")
cortas[["source", "title", "largo_content", "url"]]

## 4. Índices y planes de consulta

In [ ]:
pd.DataFrame([{"índice": nombre, "campos": info["key"], "extra": {k: v for k, v in info.items() if k not in ("key", "v")}}
              for nombre, info in db.noticias.index_information().items()])

In [ ]:
def plan(cursor):
    """Las etapas del plan ganador, por ejemplo 'LIMIT <- FETCH <- IXSCAN(recientes)'."""
    etapa, etapas = cursor.explain()["queryPlanner"]["winningPlan"], []
    while etapa:
        etapas.append(etapa["stage"] + (f"({etapa['indexName']})" if "indexName" in etapa else ""))
        etapa = etapa.get("inputStage")
    return " <- ".join(etapas)

consultas = {
    "recientes": db.noticias.find().sort("published", -1).limit(10),
    "por fuente": db.noticias.find({"source": "Hacker News"}).sort("published", -1),
    "populares": db.noticias.find({"score": {"$type": "number"}}).sort("score", -1),
    "búsqueda": db.noticias.find({"$text": {"$search": "agent"}}),
}
for nombre, cursor in consultas.items():
    p = plan(cursor)
    chequeo(f"Consulta '{nombre}' usa índice: {p}", 0 if ("IXSCAN" in p or "TEXT" in p) else 1)

## 5. Muestras

Son las mismas funciones de `hedwix_db/noticias.py` que va a usar el MCP.

In [ ]:
cols = ["published", "source", "title", "score"]
print("Últimas 10:")
pd.DataFrame(recientes(db, limite=10))[cols]

In [ ]:
print("Más populares:")
pd.DataFrame(mas_populares(db, limite=10))[cols]

In [ ]:
TEXTO = "agent"   # cambia esto para buscar otra cosa
print(f"Búsqueda: {TEXTO!r}")
pd.DataFrame(buscar(db, TEXTO, limite=10))[cols]

In [ ]:
# Una noticia completa, en el formato del funnel (como la recibiría el agent)
from pprint import pprint
item = recientes(db, fuentes=["TechCrunch AI"], limite=1) or recientes(db, limite=1)
item = dict(item[0]); item["content"] = item["content"][:600] + " […]"
pprint(item, sort_dicts=False, width=110)

## 6. Colecciones del dispatch

`usuarios` y `envios` todavía están vacías. Esta sección revisa que existan con su validador y sus índices.

In [ ]:
for nombre in ["usuarios", "envios"]:
    opciones = db.command("listCollections", filter={"name": nombre})["cursor"]["firstBatch"]
    tiene_validador = bool(opciones and "validator" in opciones[0]["options"])
    chequeo(f"{nombre}: existe con validador ({db[nombre].count_documents({})} docs, "
            f"índices {sorted(db[nombre].index_information())})", 0 if tiene_validador else 1)

## Resumen

In [ ]:
resumen = pd.DataFrame(resultados)
print(f"{resumen['ok'].sum()} de {len(resumen)} chequeos OK")
resumen.style.map(lambda ok: "background-color: #d1fae5" if ok else "background-color: #fee2e2", subset=["ok"])

In [ ]:
# Para correrlo desde la terminal: si algún chequeo falla, el notebook termina con error
assert resumen["ok"].all(), "Hay chequeos con problemas: " + ", ".join(resumen.loc[~resumen["ok"], "chequeo"])